In [ ]:
import pandas as pd
import numpy as np
import re
import random
import nltk
import json
import scipy
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import adjusted_rand_score
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from nltk.corpus import wordnet, stopwords
from nltk import pos_tag, word_tokenize

root_dir = '/home/esa/projects/'
nltk.data.path.append('/home/esa/nltk_data')

In [ ]:
df = pd.read_csv(f'{root_dir}lyrics-dataset/final_dataset_V5.csv')
df_meta = df.copy()
df_meta.drop(columns=['lyrics', 'language','new_artist_popularity', 'acousticness',
       'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness',
       'mode', 'speechiness', 'tempo', 'time_signature', 'valence', 'duration_ms'],
inplace=True)
df.drop(columns=['artist', 'song_name', 'genres', 'language', 'new_artist_popularity',
       'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness',
       'loudness', 'mode', 'speechiness', 'tempo', 'time_signature', 'valence',
       'duration_ms'],
inplace=True)

In [ ]:
artists_names = df_meta['artist_name'].unique().tolist()

In [ ]:
SECTION_RE = re.compile(r'^\[(.+?)\]\s*$', re.IGNORECASE)

def resolve_lyrics_sections_safe(text):
    if not isinstance(text, str):
        return text

    lines = text.splitlines()
    section_memory = {}
    resolved_lines = []

    current_section = None
    buffer = []
    saw_any_section = False

    def flush_section():
        nonlocal buffer, current_section
        if current_section is None:
            return

        content = "\n".join(buffer).strip()

        if content:
            section_memory[current_section] = content
            resolved_lines.append(content)
        else:
            # empty reference → reuse if exists
            if current_section in section_memory:
                resolved_lines.append(section_memory[current_section])
            else:
                # keep original label if no prior content
                resolved_lines.append(f"[{current_section}]")

        buffer.clear()

    for line in lines:
        match = SECTION_RE.match(line.strip())
        if match:
            saw_any_section = True
            flush_section()
            current_section = match.group(1).lower()
        else:
            buffer.append(line)

    flush_section()

    # If nothing resolved, return original text
    cleaned = "\n\n".join(resolved_lines).strip()

    if not cleaned:
        return text.strip()

    return cleaned


In [ ]:
df["cleaned_lyrics"] = df["lyrics"].apply(resolve_lyrics_sections_safe)
df.drop(columns=['lyrics'], inplace=True)
len(df)

In [ ]:
df.to_csv(f'{root_dir}lyrics-dataset/final_dataset_V5.csv', index=False)

In [ ]:
df = pd.read_csv(f'{root_dir}lyrics-dataset/final_dataset_V5.csv')
df_meta = pd.read_csv(f'{root_dir}lyrics-dataset/final_dataset_V5_meta.csv')
# print(df.iloc[200]["cleaned_lyrics"], df_meta.iloc[200])

In [ ]:
useless_words = {"oh", "ooh", "hoo", "na", "la", "yeah", "yea", "woah", "whoa", "ah", "ha", "hey", "mm", "hmm", "uh", "uhh", "mmm"}

def remove_useless_words(text):
    
    lines = text.split("\n")
    cleaned_lines = []
    
    for line in lines:
        words = line.split()
        if not words:
            continue

        new_words = []
        i = 0

        while i < len(words):
            word = words[i].lower()

            # Count consecutive repetitions
            j = i + 1
            while j < len(words) and words[j].lower() == word:
                j += 1

            count = j - i

            if word in useless_words:
                # Keep max 2 repetitions
                if count >= 2:
                    new_words.extend([word] * 2)
                # If single useless word surrounded by real words, drop it
                # If whole line is useless, skip later
            else:
                new_words.extend(words[i:j])

            i = j

        # Skip lines that become empty or only filler
        if any(w.lower() not in useless_words for w in new_words):
            cleaned_lines.append(" ".join(new_words))

    return "\n".join(cleaned_lines)

df["cleaned_lyrics"] = df["cleaned_lyrics"].apply(remove_useless_words)

In [ ]:
STOPWORDS = set(stopwords.words("english"))
PROTECTED = {
    "love", "hate", "sad", "happy", "lonely", "pain",
    "cry", "die", "heart", "tears",
    "not", "never", "no",
    "i", "you", "we", "me", "my", "your"
}

def clean_synonyms(word, wn_pos):
    syns = set()
    for syn in wordnet.synsets(word, pos=wn_pos):
        for lemma in syn.lemmas():
            s = lemma.name().replace("_", " ").lower()
            if s != word and s.isalpha():
                syns.add(s)
    return list(syns)

def wn_pos(tag):
    if tag.startswith("J"): return wordnet.ADJ
    if tag.startswith("R"): return wordnet.ADV
    if tag.startswith("N"): return wordnet.NOUN
    return None

def lyric_synonym_replacement(text, max_replacements=1):
    tokens = word_tokenize(text)
    tags = pos_tag(tokens)

    candidates = []
    for i, (w, t) in enumerate(tags):
        wl = w.lower()
        if (
            wl.isalpha()
            and wl not in STOPWORDS
            and wl not in PROTECTED
            and wn_pos(t)
        ):
            candidates.append((i, wl, wn_pos(t)))

    random.shuffle(candidates)

    replaced = 0
    for i, w, pos in candidates:
        if replaced >= max_replacements:
            break
        syns = clean_synonyms(w, pos)
        if syns:
            tokens[i] = random.choice(syns)
            replaced += 1

    return " ".join(tokens)

def aug(lyrics):
    lines = lyrics.split("\n")
    augmented = [
        lyric_synonym_replacement(line)
        for line in lines
    ]
    return "\n".join(augmented)

In [ ]:
df["augmented_lyrics"] = df["cleaned_lyrics"].apply(aug)
df.head()

In [ ]:
len(str(df.iloc[932]["augmented_lyrics"]).split())

In [ ]:
df.to_csv(f'{root_dir}lyrics-dataset/final_dataset_V6_(augmented).csv', index=False)

In [ ]:
baseline = np.zeros(len(df))
baseline_aug = np.zeros(len(df))

for i in range(len(df)):
    baseline[i] = len(str(df.iloc[i]["cleaned_lyrics"]).split())
    baseline_aug[i] = len(str(df.iloc[i]["augmented_lyrics"]).split())
    
ori_len = scipy.stats.mode(baseline)
aug_len = scipy.stats.mode(baseline_aug)
print(ori_len.mode)

In [ ]:
target_len = int((ori_len.mode + aug_len.mode)/2)
print(target_len)

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df["cleaned_lyrics"] + df["augmented_lyrics"])
total_words = len(tokenizer.word_index) + 1

ori_sequences = tokenizer.texts_to_sequences(df["cleaned_lyrics"])
aug_sequences = tokenizer.texts_to_sequences(df["augmented_lyrics"])

ori_padded = pad_sequences(ori_sequences, maxlen=target_len, truncating='post', padding='post')
aug_padded = pad_sequences(aug_sequences, maxlen=target_len, truncating='post', padding='post')

pairs = np.stack((ori_padded, aug_padded), axis=1)

In [ ]:
np.savez(f'{root_dir}/lyrics-dataset/paired_dataset.npz', 
         pairs=pairs,
         ori_padded=ori_padded, 
         aug_padded=aug_padded)

with open(f'{root_dir}/lyrics-dataset/tokenizer.json', 'w') as f:
    f.write(tokenizer.to_json())

In [ ]:
# To load data later
from tensorflow.keras.preprocessing.text import tokenizer_from_json

df = pd.read_csv(f'{root_dir}lyrics-dataset/clustered_data.csv')
df_meta = pd.read_csv(f'{root_dir}/lyrics-dataset/clustered_meta.csv')

loaded = np.load(f'{root_dir}/lyrics-dataset/paired_dataset.npz')
pairs = loaded['pairs']
ori_padded = loaded['ori_padded']
aug_padded = loaded['aug_padded']

with open(f'{root_dir}/lyrics-dataset/tokenizer.json', 'r') as f:
    tokenizer = tokenizer_from_json(f.read())

total_words = len(tokenizer.word_index) + 1

def get_batches(pairs, batch_size=32, ordered=False):
    if ordered:
        indices = np.arange(len(pairs))       # sequential, no shuffle
    else:
        indices = np.random.permutation(len(pairs))  # shuffle for training
    
    pairs = pairs[indices]
    
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i + batch_size]
        ori = batch[:, 0, :]
        aug = batch[:, 1, :]
        yield ori, aug

In [ ]:
def tanh_forward(x):
    out = np.tanh(x)
    cache = out
    return out, cache
def tanh_backward(dout, cache):
    dx = dout * (1 - cache ** 2)
    return dx
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
def layer_norm(x, eps=1e-8):
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)

class Embedding:
    def __init__(self, vocab_size, embed_dim):
        # uniform init works better for embeddings
        scale = np.sqrt(1.0 / embed_dim)
        self.W = np.random.uniform(-scale, scale, (vocab_size, embed_dim)).astype(np.float32)
    
    def forward(self, x):
        out = self.W[x]          # (B, T, embed_dim)
        cache = (x, self.W.shape)
        return out, cache
    
    def backward(self, dout, cache):
        x, W_shape = cache
        dW = np.zeros(W_shape)
        np.add.at(dW, x, dout)  # scatter gradients back to the touched rows
        return dW

class GlobalMaxPool:
    def __init__(self):
        pass

    def forward(self, x):
        # x: (B, T, D) → (B, D)
        out = np.max(x, axis=1)
        cache = (x, np.argmax(x, axis=1))
        return out, cache

    def backward(self, dout, cache):
        x, argmax = cache
        B, T, D = x.shape
        dx = np.zeros_like(x)
        b_idx = np.arange(B)[:, None]
        d_idx = np.arange(D)[None, :]
        dx[b_idx, argmax, d_idx] = dout  # gradient only flows through the max
        return dx

class L2Normalize:
    def __init__(self):
        pass

    def forward(self, x, eps=1e-8):
        norm = np.linalg.norm(x, axis=1, keepdims=True)
        out = x / (norm + eps)
        cache = (x, norm, eps)
        return out, cache

    def backward(self, dout, cache):
        x, norm, eps = cache
        n = norm + eps
        dot = np.sum(dout * x, axis=1, keepdims=True)
        dx = (dout - (dot / n**2) * x) / n
        return dx

class Dense:
    def __init__(self, in_channels=512, hidden=256):

        def init_w(shape):
            return np.random.randn(*shape).astype(np.float32)

        def init_b(shape):
            return np.zeros(shape, dtype=np.float32)

        scale = np.sqrt(2.0 / (in_channels + hidden))
        self.w1 = init_w((in_channels, hidden)) * scale
        self.b1 = init_b(hidden)
    
    def forward(self, x):
        out = x @ self.w1 + self.b1 # normal x*w+b
        cache = x
        return out, cache

    def backward(self, dout, cache):
        x = cache
        dx = dout @ self.w1.T
        dW = x.T @ dout
        db = dout.sum(axis=0)
        return dx, dW, db

class BiLSTM:
    def __init__(self, in_channels=32, hidden=512):

        def xavier(shape):
            scale = np.sqrt(2.0 / (shape[0] + shape[1]))
            return np.random.randn(*shape).astype(np.float32) * scale
        
        # Orthogonal for recurrent weights
        def orthogonal(shape):
            rows = max(shape[0], shape[1])
            flat = np.random.randn(rows, rows)
            u, _, vt = np.linalg.svd(flat, full_matrices=False)
            W = u if u.shape == (shape[0], shape[0]) else vt
            return W[:shape[0], :shape[1]].astype(np.float32)

        def init_b(shape):
            return np.zeros(shape, dtype=np.float32)
        
        H = hidden * 4
        self.hidden_size = hidden

        self.w1, self.w2 = xavier((in_channels, H)), orthogonal((hidden, H))
        self.b = init_b(H)
        self.b[hidden:2*hidden] = 1.0
        
        self.w1_rev, self.w2_rev = xavier((in_channels, H)), orthogonal((hidden, H))
        self.b_rev = init_b(H)
        self.b_rev[hidden:2*hidden] = 1.0
    
    def lstm_step(self, x_t, h, c, W, U, b):
        z = x_t @ W + h @ U + b
        i, f, o, g = np.split(z, 4, axis=1)
        i = sigmoid(i)
        f = sigmoid(f)
        o = sigmoid(o)
        g = np.tanh(g)
        C = f * c + i * g
        h = o * np.tanh(C)
        cache = (x_t, h, c, W, U, i, f, o, g, C)
        return h, C, cache
    
    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        
        h = np.zeros((batch_size, self.hidden_size), dtype=np.float32)
        c = np.zeros((batch_size, self.hidden_size), dtype=np.float32)
        h_rev = np.zeros((batch_size, self.hidden_size), dtype=np.float32)
        c_rev = np.zeros((batch_size, self.hidden_size), dtype=np.float32)
        
        fwd_caches, rwd_caches = [], []
        fwd_hs, rwd_hs = [], []
        
        for t in range(seq_len):
            x_t = x[:, t, :]

            h, c, cache = self.lstm_step(x_t, h, c, self.w1, self.w2, self.b)
            fwd_caches.append(cache)
            fwd_hs.append(h)
        
        for t in reversed(range(seq_len)):  # reverse direction
            x_t = x[:, t, :]
            
            h_rev, c_rev, cache = self.lstm_step(x_t, h_rev, c_rev, self.w1_rev, self.w2_rev, self.b_rev)
            rwd_caches.append(cache)
            rwd_hs.append(h_rev)
        
        rwd_caches.reverse()
        rwd_hs.reverse()

        out = np.concatenate( # combine both directions
            [np.stack(fwd_hs, axis=1), np.stack(rwd_hs, axis=1)], axis=2
        )
        cache = (fwd_caches, rwd_caches, x.shape)
        
        return out, cache
    
    def lstm_cell_backward(self, dh_next, dc_next, cache):
        x_t, h_prev, c_prev, Wx, Wh, i, f, o, g, c_next = cache

        tanh_c = np.tanh(c_next)

        # output gate and cell
        do = dh_next * tanh_c
        dc = dh_next * o * (1 - tanh_c ** 2) + dc_next

        # gates
        di = dc * g
        df = dc * c_prev
        dg = dc * i
        dc_prev = dc * f

        # gate pre-activations (before sigmoid/tanh)
        di_raw = di * i * (1 - i)       # sigmoid backward
        df_raw = df * f * (1 - f)
        do_raw = do * o * (1 - o)
        dg_raw = dg * (1 - g ** 2)      # tanh backward

        dz = np.concatenate([di_raw, df_raw, do_raw, dg_raw], axis=1)  # (B, 4H)

        dx_t   = dz @ Wx.T              # (B, input_dim)
        dh_prev = dz @ Wh.T             # (B, H)
        dWx    = x_t.T @ dz             # (input_dim, 4H)
        dWh    = h_prev.T @ dz          # (H, 4H)
        db     = dz.sum(axis=0)         # (4H,)

        return dx_t, dh_prev, dc_prev, dWx, dWh, db
    
    def backward(self,dout, cache):
        fwd_caches, bwd_caches, x_shape = cache
        B, T, input_dim = x_shape

        # split dout into forward and reverse
        H = dout.shape[2] // 2
        dout_fwd = dout[:, :, :H]
        dout_bwd = dout[:, :, H:]

        dx = np.zeros(x_shape, dtype=np.float32)

        # accumulate weight grads
        dfWx = np.zeros_like(fwd_caches[0][3])  # Wx shape from cache
        dfWh = np.zeros_like(fwd_caches[0][4])
        dfb  = np.zeros_like(fwd_caches[0][4][:H])  # H, not 4H — fix below

        # easier to just init from the actual shapes
        dfWx = np.zeros((input_dim, H * 4), dtype=np.float32)
        dfWh = np.zeros((H, H * 4), dtype=np.float32)
        dfb  = np.zeros(H * 4, dtype=np.float32)

        dbWx = np.zeros_like(dfWx)
        dbWh = np.zeros_like(dfWh)
        dbb  = np.zeros_like(dfb)

        # reverse (now in reverse, so forwards)
        dh = np.zeros((B, H), dtype=np.float32)
        dc = np.zeros((B, H), dtype=np.float32)

        for t in reversed(range(T)):
            dh += dout_fwd[:, t, :]   # add gradient from output at this timestep
            dx_t, dh, dc, dWx_t, dWh_t, db_t = self.lstm_cell_backward(dh, dc, fwd_caches[t])
            dx[:, t, :] += dx_t
            dfWx += dWx_t
            dfWh += dWh_t
            dfb  += db_t

        # forwards (now in reverse)
        dh = np.zeros((B, H), dtype=np.float32)
        dc = np.zeros((B, H), dtype=np.float32)

        for t in range(T):
            dh += dout_bwd[:, t, :]
            dx_t, dh, dc, dWx_t, dWh_t, db_t = self.lstm_cell_backward(dh, dc, bwd_caches[t])
            dx[:, t, :] += dx_t
            dbWx += dWx_t
            dbWh += dWh_t
            dbb  += db_t

        grads = {
            'lstm_fWx': dfWx, 'lstm_fWh': dfWh, 'lstm_fb': dfb,
            'lstm_bWx': dbWx, 'lstm_bWh': dbWh, 'lstm_bb': dbb,
        }
        return dx, grads

def clip_by_global_norm(grads, max_norm=1.0):
    total_norm = np.sqrt(sum(np.sum(g**2) for g in grads.values()))
    if total_norm > max_norm:
        scale = max_norm / (total_norm + 1e-6)
        grads = {k: v * scale for k, v in grads.items()}
    return grads
        
class Adam:
    def __init__(self, lr=1e-4, b1=0.9, b2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps = lr, b1, b2, eps
        self.t = 0
        self.m = {}
        self.v = {}

    def step(self, params, grads):
        self.t += 1
        for k in params:
            if k not in self.m:  # lazy init on first step
                self.m[k] = np.zeros_like(params[k])
                self.v[k] = np.zeros_like(params[k])

            self.m[k] = self.b1*self.m[k] + (1-self.b1)*grads[k]
            self.v[k] = self.b2*self.v[k] + (1-self.b2)*(grads[k]**2)

            m_hat = self.m[k] / (1 - self.b1**self.t)
            v_hat = self.v[k] / (1 - self.b2**self.t)

            params[k] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


In [ ]:
EmbeddingLayer = Embedding(total_words, 32)
BiLSTMLayer = BiLSTM(in_channels=32, hidden=512)
PoolLayer = GlobalMaxPool()
Dense1 = Dense(in_channels=1024, hidden=256)
Dense2 = Dense(in_channels=256, hidden=64)
NormLayer = L2Normalize()
optimizer = Adam(lr=0.0002)
num_epochs = 5

In [ ]:
pairs.shape

In [ ]:
type(pairs)

In [ ]:
for ori_batch, aug_batch in get_batches(pairs):
    print(ori_batch.shape, aug_batch.shape)  # check if shape is correct (32, 199)
    break

In [ ]:
def backwards(cache, dz):
    emb_cache, lstm_cache, pool_cache, d1_cache, tanh_cache, d2_cache, norm_cache = cache
    
    d = NormLayer.backward(dz, norm_cache)
    d, dw2, db2 = Dense2.backward(d, d2_cache)
    d = tanh_backward(d, tanh_cache)
    d, dw1, db1 = Dense1.backward(d, d1_cache)
    d = PoolLayer.backward(d, pool_cache)
    d, dlstm_grads = BiLSTMLayer.backward(d, lstm_cache)
    demb = EmbeddingLayer.backward(d, emb_cache)
    
    grads = {
        'embedding': demb,
        'lstm_fWx': dlstm_grads['lstm_fWx'], 'lstm_fWh': dlstm_grads['lstm_fWh'], 'lstm_fb': dlstm_grads['lstm_fb'],
        'lstm_rWx': dlstm_grads['lstm_bWx'], 'lstm_rWh': dlstm_grads['lstm_bWh'], 'lstm_rb': dlstm_grads['lstm_bb'],
        'd1_w': dw1, 'd1_b': db1,
        'd2_w': dw2, 'd2_b': db2
    }
    
    return grads

def contrastive_loss(z1, z2, temperature=0.9):
    B = z1.shape[0]
    
    # concat all vectors and compute full similarity matrix
    z = np.concatenate([z1, z2], axis=0)
    sim = (z @ z.T) / temperature

    # mask out self-similarity
    np.fill_diagonal(sim, -np.inf)

    # true positive pairs
    labels = np.concatenate([np.arange(B, 2*B), np.arange(0, B)])  # (2B,)

    # cross entropy loss
    sim_max = sim.max(axis=1, keepdims=True)
    sim_shifted = sim - sim_max

    exp_sim = np.exp(sim_shifted)
    exp_sim[np.isinf(sim)] = 0
    sum_exp = exp_sim.sum(axis=1, keepdims=True)
    probs = exp_sim / sum_exp

    # loss
    pos_probs = probs[np.arange(2*B), labels]
    pos_probs = np.clip(pos_probs, 1e-8, None)
    loss = -np.log(pos_probs).mean()

    # gradient
    dprobs = probs.copy()
    dprobs[np.arange(2*B), labels] -= 1
    dprobs /= (2 * B)
    dz = (dprobs + dprobs.T) @ z / temperature
    
    dz1 = dz[:B]
    dz2 = dz[B:]

    return loss, dz1, dz2

def train_encoder(z1, z2):
    
    # Forward pass
    ori_emb, ori_emb_cache = EmbeddingLayer.forward(z1)
    aug_emb, aug_emb_cache = EmbeddingLayer.forward(z2)

    ori_lstm, ori_lstm_cache = BiLSTMLayer.forward(ori_emb)
    aug_lstm, aug_lstm_cache = BiLSTMLayer.forward(aug_emb)
    ori_lstm = layer_norm(ori_lstm)
    aug_lstm = layer_norm(aug_lstm)
    # print(f"lstm out norm:  {np.linalg.norm(ori_lstm, axis=(1,2)).mean():.4f}, lstm sample var: {ori_lstm.var(axis=(1,2)).mean():.6f}")
    
    ori_pool, ori_pool_cache = PoolLayer.forward(ori_lstm)
    aug_pool, aug_pool_cache = PoolLayer.forward(aug_lstm)
    # print(f"pool out norm:  {np.linalg.norm(ori_pool, axis=1).mean():.4f}, pool sample var: {ori_pool.var(axis=1).mean():.6f}")

    ori_d1, ori_d1_cache = Dense1.forward(ori_pool)
    aug_d1, aug_d1_cache = Dense1.forward(aug_pool)
    # print(f"d1 out norm:    {np.linalg.norm(ori_d1, axis=1).mean():.4f}")
    ori_d1_act, ori_tanh_cache = tanh_forward(ori_d1)
    aug_d1_act, aug_tanh_cache = tanh_forward(aug_d1)
    # print(f"d1 act norm:    {np.linalg.norm(ori_d1_act, axis=1).mean():.4f}")

    ori_d2, ori_d2_cache = Dense2.forward(ori_d1_act)
    aug_d2, aug_d2_cache = Dense2.forward(aug_d1_act)
    # print(f"d2 out norm:    {np.linalg.norm(ori_d2, axis=1).mean():.4f}")

    ori_norm, ori_norm_cache = NormLayer.forward(ori_d2)
    aug_norm, aug_norm_cache = NormLayer.forward(aug_d2)
    # print(f"norm out:  {ori_norm.mean():.4f}")
    # print(f"norm out norm:  {np.linalg.norm(ori_norm, axis=1).mean():.4f}")
    
    ori_caches = (ori_emb_cache, ori_lstm_cache, ori_pool_cache, ori_d1_cache, ori_tanh_cache, ori_d2_cache, ori_norm_cache)
    aug_caches = (aug_emb_cache, aug_lstm_cache, aug_pool_cache, aug_d1_cache, aug_tanh_cache, aug_d2_cache, aug_norm_cache)

    # print(f"z1 has nan: {np.isnan(z1).any()}")
    # print(f"z1 has inf: {np.isinf(z1).any()}")
    # print(f"z1 norm mean: {np.linalg.norm(z1, axis=1).mean():.4f}")
    
    # Calculate contrastive loss and grads
    loss, dz1, dz2 = contrastive_loss(ori_norm, aug_norm)
    
    grads_ori = backwards(ori_caches, dz1)
    grads_aug = backwards(aug_caches, dz2)
    grads = {k: grads_ori[k] + grads_aug[k] for k in grads_ori}
    # grads = clip_by_global_norm(grads, max_norm=1.0)
    
    # Backpropagate and update parameters
    params = {
        'embedding': EmbeddingLayer.W,
        'lstm_fWx': BiLSTMLayer.w1, 'lstm_fWh': BiLSTMLayer.w2, 'lstm_fb': BiLSTMLayer.b,
        'lstm_rWx': BiLSTMLayer.w1_rev, 'lstm_rWh': BiLSTMLayer.w2_rev, 'lstm_rb': BiLSTMLayer.b_rev,
        'd1_w': Dense1.w1, 'd1_b': Dense1.b1,
        'd2_w': Dense2.w1, 'd2_b': Dense2.b1,
    }
    
    # for k, g in grads.items():
    #     print(f"{k}: grad_mean={g.mean():.6f}, grad_std={g.std():.6f}")
    
    optimizer.step(params, grads)
    
    EmbeddingLayer.W = params['embedding']
    BiLSTMLayer.w1, BiLSTMLayer.w2, BiLSTMLayer.b = params['lstm_fWx'], params['lstm_fWh'], params['lstm_fb']
    BiLSTMLayer.w1_rev, BiLSTMLayer.w2_rev, BiLSTMLayer.b_rev = params['lstm_rWx'], params['lstm_rWh'], params['lstm_rb']
    Dense1.w1 = params['d1_w'];  Dense1.b1 = params['d1_b']
    Dense2.w1 = params['d2_w'];  Dense2.b1 = params['d2_b']
    
    return loss


In [ ]:
# Full training loop
for epoch in tqdm(range(num_epochs)):
    total_loss = 0
    
    for ori_batch, aug_batch in get_batches(pairs, ordered=False):
        total_loss += train_encoder(ori_batch, aug_batch)
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [ ]:
# Test training loop
# optimizer = Adam(lr=0.002)
total_loss = 0
count = 0
epoch = 0

for ori_batch, aug_batch in get_batches(pairs, ordered=False):
    total_loss += train_encoder(ori_batch, aug_batch)
    count += 1
    if count >= 40:
        break

print(f"Loss: {total_loss:.4f}")
# print(Dense1.w1.mean(), Dense1.w1.std())
# print(Dense2.w1.mean(), Dense2.w1.std())
# print(BiLSTMLayer.w1.mean())

In [ ]:
def save_model(path, emb, lstm, D1, D2):
    np.savez(path,
        emb_W=emb.W,
        lstm_fWx=lstm.w1, lstm_fWh=lstm.w2, lstm_fb=lstm.b,
        lstm_bWx=lstm.w1_rev, lstm_bWh=lstm.w2_rev, lstm_bb=lstm.b_rev,
        d1_w=D1.w1, d1_b=D1.b1,
        d2_w=D2.w1, d2_b=D2.b1,
    )

def load_model(path, vocab_size, embed_dim=32, hidden_dim=512):
    d = np.load(path)

    emb = Embedding(vocab_size, embed_dim)
    emb.W = d["emb_W"]

    lstm = BiLSTM(in_channels=embed_dim, hidden=hidden_dim)
    lstm.w1, lstm.w2, lstm.b = d["lstm_fWx"], d["lstm_fWh"], d["lstm_fb"]
    lstm.w1_rev, lstm.w2_rev, lstm.b_rev = d["lstm_bWx"], d["lstm_bWh"], d["lstm_bb"]

    d1 = Dense(in_channels=hidden_dim * 2, hidden=256)
    d1.w1, d1.b1 = d["d1_w"], d["d1_b"]

    d2 = Dense(in_channels=256, hidden=64)
    d2.w1, d2.b1 = d["d2_w"], d["d2_b"]

    return emb, lstm, d1, d2

In [ ]:
save_model(f'{root_dir}/Skripsi/models(2)/encoder.npz', EmbeddingLayer, BiLSTMLayer, Dense1, Dense2)

LATESET HERE ==================================================================================================================================================================

In [ ]:
Embedding_untrained = Embedding(total_words, 32)
BiLSTM_untrained = BiLSTM(in_channels=32, hidden=512)
Dense1_untrained = Dense(in_channels=1024, hidden=256)
Dense2_untrained = Dense(in_channels=256, hidden=64)

# EmbeddingLayer, BiLSTMLayer, Dense1, Dense2 = load_model(f'{root_dir}/Skripsi/models/encoder.npz', total_words, embed_dim=32, hidden_dim=512)

# PoolLayer = GlobalMaxPool()
# NormLayer = L2Normalize()
# optimizer = Adam(lr=0.0002)

In [ ]:
all_embeddings = df['embedding'].apply(lambda s: np.fromstring(s.strip("[]"), sep=" ")).tolist()
all_embeddings = np.array(all_embeddings)
best_centroids = np.load(f'{root_dir}/Skripsi/models/centroids.npz')['centroids']
dists = np.linalg.norm(all_embeddings[:, None] - best_centroids[None, :], axis=2)
cluster_labels = np.argmin(dists, axis=1)
best_k = best_centroids.shape[0]

MODEL TESTING

In [ ]:
def encode(x, emb, lstm, D1, D2):
    emb_out, _  = emb.forward(x)
    lstm_out, _ = lstm.forward(emb_out)
    pool_out, _ = PoolLayer.forward(lstm_out)
    d1_out, _   = D1.forward(pool_out)
    d1_act, _   = tanh_forward(d1_out)
    d2_out, _   = D2.forward(d1_act)
    norm_out, _ = NormLayer.forward(d2_out)
    return norm_out  # (B, 64)

In [ ]:
# grab one batch
batch_gen = get_batches(pairs, ordered=False)
ori_batch, aug_batch = next(batch_gen)

ori_vecs = encode(ori_batch, EmbeddingLayer, BiLSTMLayer, Dense1, Dense2)
aug_vecs = encode(aug_batch, EmbeddingLayer, BiLSTMLayer, Dense1, Dense2)

# ori_vecs = encode(ori_batch, Embedding_untrained, BiLSTM_untrained, Dense1_untrained, Dense2_untrained)
# aug_vecs = encode(aug_batch, Embedding_untrained, BiLSTM_untrained, Dense1_untrained, Dense2_untrained)

# for each sample, compute distance to its positive and all negatives
for i in range(len(ori_vecs)):
    anchor   = ori_vecs[i]       # the original
    positive = aug_vecs[i]       # its augmented pair

    pos_dist = np.linalg.norm(anchor - positive)

    # distance to all other aug samples (negatives)
    neg_dists = [np.linalg.norm(anchor - aug_vecs[j]) 
                 for j in range(len(aug_vecs)) if j != i]

    avg_neg_dist = np.mean(neg_dists)

    print(f"Sample {i:2d} | pos_dist: {pos_dist:.4f} | avg_neg_dist: {avg_neg_dist:.4f} | {'✓' if pos_dist < avg_neg_dist else '✗'}")

wins = sum(
    np.linalg.norm(ori_vecs[i] - aug_vecs[i]) < 
    np.mean([np.linalg.norm(ori_vecs[i] - aug_vecs[j]) 
             for j in range(len(aug_vecs)) if j != i])
    for i in range(len(ori_vecs))
)
print(f"Positive closer than avg negative: {wins}/{len(ori_vecs)}")

In [ ]:
# Evaluate separation of trained model on whole dataset
total_wins = 0
total_samples = 0

for ori_batch, aug_batch in get_batches(pairs, ordered=False):
    ori_vecs = encode(ori_batch, EmbeddingLayer, BiLSTMLayer, Dense1, Dense2)
    aug_vecs = encode(aug_batch, EmbeddingLayer, BiLSTMLayer, Dense1, Dense2)
    
    for i in range(len(ori_vecs)):
        pos_dist = np.linalg.norm(ori_vecs[i] - aug_vecs[i])
        neg_dists = [np.linalg.norm(ori_vecs[i] - aug_vecs[j]) 
                     for j in range(len(aug_vecs)) if j != i]
        if pos_dist < np.mean(neg_dists):
            total_wins += 1
        total_samples += 1

print(f"Overall: {total_wins}/{total_samples} ({100*total_wins/total_samples:.1f}%)")

In [ ]:
# Evaluate separation of untrained model on whole dataset
total_wins = 0
total_samples = 0

for ori_batch, aug_batch in get_batches(pairs, ordered=False):
    ori_vecs = encode(ori_batch, Embedding_untrained, BiLSTM_untrained, Dense1_untrained, Dense2_untrained)
    aug_vecs = encode(aug_batch, Embedding_untrained, BiLSTM_untrained, Dense1_untrained, Dense2_untrained)

    for i in range(len(ori_vecs)):
        pos_dist = np.linalg.norm(ori_vecs[i] - aug_vecs[i])
        neg_dists = [np.linalg.norm(ori_vecs[i] - aug_vecs[j]) 
                     for j in range(len(aug_vecs)) if j != i]
        if pos_dist < np.mean(neg_dists):
            total_wins += 1
        total_samples += 1

print(f"Overall: {total_wins}/{total_samples} ({100*total_wins/total_samples:.1f}%)")

In [ ]:
failures = []
for ori_batch, aug_batch in get_batches(pairs, ordered=True):
    ori_vecs = encode(ori_batch, EmbeddingLayer, BiLSTMLayer, Dense1, Dense2)
    aug_vecs = encode(aug_batch, EmbeddingLayer, BiLSTMLayer, Dense1, Dense2)
    
    for i in range(len(ori_vecs)):
        pos_dist = np.linalg.norm(ori_vecs[i] - aug_vecs[i])
        neg_dists = [np.linalg.norm(ori_vecs[i] - aug_vecs[j]) 
                     for j in range(len(aug_vecs)) if j != i]
        avg_neg = np.mean(neg_dists)
        if pos_dist > avg_neg:
            failures.append(pos_dist - avg_neg)  # how badly it failed

print(f"Failures: {len(failures)}")
print(f"Mean failure margin: {np.mean(failures):.4f}")
print(f"Max failure margin:  {np.max(failures):.4f}")

THE CLUSTERING PART

In [ ]:
# generate embeddings for all samples for clustering
all_embeddings = []

for ori, _ in get_batches(pairs, batch_size=32, ordered=True):
    vec = encode(ori, EmbeddingLayer, BiLSTMLayer, Dense1, Dense2)
    all_embeddings.append(vec)

all_embeddings = np.concatenate(all_embeddings, axis=0)  # (3000, 64)
print(all_embeddings.shape)

In [ ]:
# determine centroids for each number of clusters with KMeans
def KMeans_manual(C,Z):
    # C: number of clusters, Z: embeddings
    N, D = Z.shape
    # Initialize centroids randomly from data points
    # np.random.seed(0)
    centroids = Z[np.random.choice(N, C, replace=False)]
    wcss = np.inf

    for _ in range(100):
        # Assign points to nearest centroid
        dists = np.linalg.norm(Z[:, None] - centroids[None, :], axis=2)  # (N, C)
        closest = np.argmin(dists, axis=1)  # (N,)

        # Update centroids
        new_centroids = np.array([Z[closest == i].mean(axis=0) for i in range(C)])
        
        # Check for convergence
        if np.allclose(centroids, new_centroids):
            break
        
        centroids = new_centroids
    
    wcss = sum((np.linalg.norm(Z[closest == i] - centroids[i], axis=1) ** 2).sum() for i in range(C))
    
    return wcss, centroids

# determine optimal k using elbow method
def find_elbow(K_range, inertias):
    K = np.array(list(K_range))
    inertias = np.array(inertias)
    
    # normalize both axes
    K_norm = (K - K.min()) / (K.max() - K.min())
    I_norm = (inertias - inertias.min()) / (inertias.max() - inertias.min())
    
    # draw line from first to last point
    p1 = np.array([K_norm[0], I_norm[0]])
    p2 = np.array([K_norm[-1], I_norm[-1]])
    
    # distance from each point to the line
    line_vec = p2 - p1
    line_len = np.linalg.norm(line_vec)
    
    dists = []
    for i in range(len(K)):
        p = np.array([K_norm[i], I_norm[i]])
        d = np.abs(np.cross(line_vec, p1 - p)) / line_len
        dists.append(d)
    
    # farthest point from the line is the elbow
    elbow_idx = np.argmax(dists)
    return K[elbow_idx]

In [ ]:
inertias = []
centroids_list = []
K_range = range(2, 30)

for k in K_range:
    wcss, centroids = KMeans_manual(k, all_embeddings)
    inertias.append(wcss)
    centroids_list.append(centroids)
    # print(f"k={k}, wcss={wcss:.2f}")

best_k = find_elbow(K_range, inertias)
best_centroids = centroids_list[K_range.index(best_k)]
dists = np.linalg.norm(all_embeddings[:, None] - best_centroids[None, :], axis=2)
cluster_labels = np.argmin(dists, axis=1)

print(f"Optimal number of clusters: {best_k}")
for i in range(best_k):
    print(f"Cluster {i}: {(cluster_labels == i).sum()} samples")

plt.plot(list(K_range), inertias, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.xticks(list(K_range))
plt.grid(True)
plt.show()

In [ ]:
df.drop(columns=['embedding'], inplace=True)
df.drop(columns=['cluster'], inplace=True)
df_meta.drop(columns=['cluster'], inplace=True)

df['cluster'] = cluster_labels
df['embedding'] = list(all_embeddings)
df_meta['cluster'] = cluster_labels

df.to_csv(f'{root_dir}lyrics-dataset/clustered_data.csv', index=False)
df_meta.to_csv(f'{root_dir}lyrics-dataset/clustered_meta.csv', index=False)
np.savez(f'{root_dir}/Skripsi/models/centroids.npz', centroids=best_centroids)

In [ ]:
df2 = df.copy()
df_meta2 = df_meta.copy()

df2.drop(columns=['embedding'], inplace=True)
df2.drop(columns=['cluster'], inplace=True)
df_meta2.drop(columns=['cluster'], inplace=True)

df2['cluster'] = cluster_labels
df2['embedding'] = list(all_embeddings)
df_meta2['cluster'] = cluster_labels

df2.to_csv(f'{root_dir}lyrics-dataset/clustered_data_2.csv', index=False)
df_meta2.to_csv(f'{root_dir}lyrics-dataset/clustered_meta_2.csv', index=False)
np.savez(f'{root_dir}/Skripsi/models(2)/centroids.npz', centroids=best_centroids)

In [ ]:
print("Cluster 3:")
# print(df[cluster_labels == 3]['cleaned_lyrics'].values)
print(df_meta[cluster_labels == 3]['genres'].values)
print(df_meta[cluster_labels == 3]['song_name'].values)
print(df_meta[cluster_labels == 3]['artist_name'].values)

In [ ]:
for c in [3]:
    print(f"\nCluster {c}:")
    print(df_meta[cluster_labels == c]['genres'].value_counts())
    print(df_meta[cluster_labels == c]['artist_name'].values)

In [ ]:
def silhouette_score(embeddings, labels):
    
    N = len(embeddings)
    clusters = np.unique(labels)
    a = np.zeros(N)
    b = np.full(N, np.inf)

    # full pairwise distance matrix at once
    diff = embeddings[:, None, :] - embeddings[None, :, :]
    dist_matrix = np.linalg.norm(diff, axis=2)

    for c in clusters:
        mask = labels == c
        idx = np.where(mask)[0]
        n_c = len(idx)

        if n_c > 1:
            intra = dist_matrix[np.ix_(idx, idx)]
            a[idx] = intra.sum(axis=1) / (n_c - 1)  # exclude self

        for c2 in clusters:
            if c2 == c:
                continue
            mask2 = labels == c2
            idx2 = np.where(mask2)[0]
            inter = dist_matrix[np.ix_(idx, idx2)]
            mean_inter = inter.mean(axis=1)
            b[idx] = np.minimum(b[idx], mean_inter)

    scores = (b - a) / np.maximum(a, b)
    return scores.mean()

def davies_bouldin_score(embeddings, labels, centroids=None):
    clusters = np.unique(labels)
    K = len(clusters)

    # centroid of each cluster
    if centroids is None:
        centroids = np.array([embeddings[labels == c].mean(axis=0) for c in clusters])

    # scatter: mean distance from points to their centroid
    scatter = np.array([
        np.linalg.norm(embeddings[labels == c] - centroids[i], axis=1).mean()
        for i, c in enumerate(clusters)
    ])

    db_sum = 0
    for i in range(K):
        max_ratio = -np.inf
        for j in range(K):
            if i == j:
                continue
            # distance between centroids
            centroid_dist = np.linalg.norm(centroids[i] - centroids[j])
            ratio = (scatter[i] + scatter[j]) / centroid_dist
            if ratio > max_ratio:
                max_ratio = ratio
        db_sum += max_ratio

    return db_sum / K

def cluster_stability(embeddings, k, n_runs=10, subsample_ratio=0.8):
    N = len(embeddings)
    labels_runs = []

    for _ in range(n_runs):
        idx = np.random.choice(N, int(N * subsample_ratio), replace=False)
        subset = embeddings[idx]
        _, centroids = KMeans_manual(k, subset)

        # assign full dataset to these centroids
        dists = np.linalg.norm(embeddings[:, None] - centroids[None, :], axis=2)
        labels = np.argmin(dists, axis=1)
        labels_runs.append(labels)

    # compare all pairs of runs using ARI
    ari_scores = []
    for i in range(len(labels_runs)):
        for j in range(i+1, len(labels_runs)):
            ari_scores.append(adjusted_rand_score(labels_runs[i], labels_runs[j]))

    mean_ari = np.mean(ari_scores)
    std_ari  = np.std(ari_scores)
    return mean_ari, std_ari

In [ ]:
sil = silhouette_score(all_embeddings, cluster_labels)
db = davies_bouldin_score(all_embeddings, cluster_labels, best_centroids)
sta_mean, sta_std = cluster_stability(all_embeddings, k=best_k)

# npz = np.load(f'{root_dir}/Skripsi/cluster_metrics.npz')
# sil = npz['silhouette']
# db = npz['davies_bouldin']
# sta_mean = npz['stability_mean']
# sta_std = npz['stability_std']

print(f"Silhouette Score: {sil:.4f}")
print(f"Davies-Bouldin Index: {db:.4f}")
print(f"Cluster Stability — Mean ARI: {sta_mean:.4f}, Std: {sta_std:.4f}")

In [ ]:
np.savez(
    f'{root_dir}/Skripsi/cluster_metrics_2.npz', 
    silhouette=sil, 
    davies_bouldin=db, 
    stability_mean=sta_mean, 
    stability_std=sta_std
)

In [ ]:
untrained_embeddings = []

for ori_batch, _ in get_batches(pairs, batch_size=32, ordered=True):
    vecs = encode(ori_batch, Embedding_untrained, BiLSTM_untrained, Dense1_untrained, Dense2_untrained)
    untrained_embeddings.append(vecs)

untrained_embeddings = np.concatenate(untrained_embeddings, axis=0)

In [ ]:
untrained_inertias = []
untrained_centroids_list = []
K_range = range(2, 30)

for k in K_range:
    wcss, centroids = KMeans_manual(k, untrained_embeddings)
    untrained_inertias.append(wcss)
    untrained_centroids_list.append(centroids)
    # print(f"k={k}, wcss={wcss:.2f}")

untrained_best_k = find_elbow(K_range, untrained_inertias)
untrained_best_centroids = untrained_centroids_list[K_range.index(untrained_best_k)]
untrained_dists = np.linalg.norm(untrained_embeddings[:, None] - untrained_best_centroids[None, :], axis=2)
untrained_cluster_labels = np.argmin(untrained_dists, axis=1)

In [ ]:
untrained_sil = silhouette_score(untrained_embeddings, untrained_cluster_labels)
untrained_db = davies_bouldin_score(untrained_embeddings, untrained_cluster_labels, untrained_best_centroids)
untrained_sta_mean, untrained_sta_std = cluster_stability(untrained_embeddings, k=untrained_best_k)

print(f"Silhouette Score: {untrained_sil:.4f}")
print(f"Davies-Bouldin Index: {untrained_db:.4f}")
print(f"Cluster Stability — Mean ARI: {untrained_sta_mean:.4f}, Std: {untrained_sta_std:.4f}")

In [ ]:
def sample_pairwise_dists(embeddings, n_samples=500):
    idx = np.random.choice(len(embeddings), n_samples, replace=False)
    subset = embeddings[idx]
    dists = []
    for i in range(len(subset)):
        for j in range(i+1, len(subset)):
            dists.append(np.linalg.norm(subset[i] - subset[j]))
    return np.array(dists)

trained_dists   = sample_pairwise_dists(all_embeddings)
untrained_dists = sample_pairwise_dists(untrained_embeddings)

plt.figure(figsize=(8, 4))
plt.hist(untrained_dists, bins=50, alpha=0.5, label='Untrained', color='gray')
plt.hist(trained_dists,   bins=50, alpha=0.5, label='Trained',   color='steelblue')
plt.xlabel('Pairwise Distance')
plt.ylabel('Frequency')
plt.title('Pairwise Distance Distribution: Trained vs Untrained')
plt.legend()
plt.show()

In [ ]:
batch_gen = get_batches(pairs)
ori_batch, aug_batch = next(batch_gen)

results = {}
for k in ["trained", "untrained"]:
    avg_neg_dists = []
    avg_pos_dists = []
    
    if k == "trained":
        ori_vecs = encode(ori_batch, EmbeddingLayer, BiLSTMLayer, Dense1, Dense2)
        aug_vecs = encode(aug_batch, EmbeddingLayer, BiLSTMLayer, Dense1, Dense2)
    else:
        ori_vecs = encode(ori_batch, Embedding_untrained, BiLSTM_untrained, Dense1_untrained, Dense2_untrained)
        aug_vecs = encode(aug_batch, Embedding_untrained, BiLSTM_untrained, Dense1_untrained, Dense2_untrained)

    # for each sample, compute distance to its positive and all negatives
    for i in range(len(ori_vecs)):
        anchor   = ori_vecs[i]       # the original
        positive = aug_vecs[i]       # its augmented pair

        pos_dist = np.linalg.norm(anchor - positive)
        avg_pos_dists.append(pos_dist)

        neg_dists = [np.linalg.norm(anchor - aug_vecs[j])
                     for j in range(len(aug_vecs)) if j != i]
        avg_neg_dists.append(np.mean(neg_dists))

    results[k] = {
        "pos": np.mean(avg_pos_dists),
        "neg": np.mean(avg_neg_dists),
        "pos_all": np.array(avg_pos_dists),
        "neg_all": np.array(avg_neg_dists),
    }

labels = ["Trained", "Untrained"]
pos_values = [results["trained"]["pos"], results["untrained"]["pos"]]
neg_values = [results["trained"]["neg"], results["untrained"]["neg"]]

x = np.arange(len(labels))
width = 0.35

# plt.figure(figsize=(8, 5))
# plt.bar(x - width/2, pos_values, width, label="Average positive distance", color="tab:blue")
# plt.bar(x + width/2, neg_values, width, label="Average negative distance", color="tab:orange")
# plt.xticks(x, labels)
# plt.ylabel("Distance")
# plt.title("Average positive vs negative distances for trained and untrained encoders")
# plt.legend()
# plt.tight_layout()
# plt.show()

# Optional distribution plot for deeper comparison
plt.figure(figsize=(8, 5))
plt.hist(results["trained"]["pos_all"], bins=30, alpha=0.6, label="Trained positive", color="tab:blue")
plt.hist(results["trained"]["neg_all"], bins=30, alpha=0.3, label="Trained negative", color="tab:cyan")
plt.hist(results["untrained"]["pos_all"], bins=30, alpha=0.6, label="Untrained positive", color="tab:orange")
plt.hist(results["untrained"]["neg_all"], bins=30, alpha=0.3, label="Untrained negative", color="tab:red")
plt.xlabel("Distance")
plt.ylabel("Frequency")
plt.title("Distance distributions for trained and untrained encoders")
plt.legend()
plt.tight_layout()
plt.show()